# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Enter Name]
**Student ID:** [Enter ID]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [2]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [4]:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content, response.usage


answer, usage = ask_llm("What is microfinance?")

print(answer)
print(usage)

Microfinance refers to the provision of financial services, such as loans, savings, and insurance, to low-income individuals or groups who do not have access to traditional banking services. The goal of microfinance is to provide opportunities for economic empowerment and poverty reduction by giving people the financial tools they need to start or expand small businesses, invest in their education or health, or improve their living conditions.

Microfinance typically involves small loan amounts, often with favorable interest rates and repayment terms, and is designed to be accessible to people who may not qualify for traditional bank loans due to lack of collateral, credit history, or other requirements. Microfinance institutions (MFIs) may also offer other financial services, such as savings accounts, remittances, and insurance products.

Microfinance has been recognized as an effective way to promote economic development, reduce poverty, and improve living standards in developing cou

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** [

1. What is the difference between the system and user roles?
The system role gives the AI instructions on how it should behave or respond. For example, the system message could say, “You are a helpful financial assistant.” The user role contains the actual question or task, such as “Explain microfinance in simple terms.”

2. What is a token, roughly? Why do API providers bill per token rather than per request?
A token is a small piece of text, which can be a whole word or part of a word. API providers charge per token because longer inputs and responses require more computing power to process. This makes token-based pricing more accurate than charging the same amount for every request.]

### Part 1.2 — Temperature: the randomness dial

In [5]:
question = "Suggest a name for a savings product for market traders in Accra."

print("Temperature = 0.0")
for i in range(5):
    answer, usage = ask_llm(question, temperature=0.0)
    print(i + 1, answer)

print("\nTemperature = 1.2")
for i in range(5):
    answer, usage = ask_llm(question, temperature=1.2)
    print(i + 1, answer)

Temperature = 0.0
1 Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders.
5. **Sika Kurom**: "Sika" means "money" in Ghanaian, and "Kurom" means "box" or "container", so this name suggests a safe and secure place to store savings.
6. **Traders' Fund**: This name is straightforward and emphasizes the idea of a collective fund for market traders.
7. **Adanfo Save**: "Adanfo" means "friends" or "partners" in Ghanaian, so this name suggests a sense of community and cooperation.

Choose the one that resonat

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** [At temperature 0.0, the answers were very similar and some were exactly the same, with only small changes in a few of the responses. At temperature 1.2, the answers were more varied and gave different product names and explanations. For the loan decision-support system, I would use a low temperature such as 0 because the system needs to give consistent and reliable results rather than creative or unpredictable answers.]

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [7]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [10]:
SUMMARY_PROMPT_V1 = lambda letter: f"Summarize this:\n\n{letter}"

print("L002:")
answer, usage = ask_llm(SUMMARY_PROMPT_V1(LETTERS["L002"]))
print(answer)

print("\nL006:")
answer, usage = ask_llm(SUMMARY_PROMPT_V1(LETTERS["L006"]))
print(answer)

SUMMARY_SYSTEM = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Summarize loan applications factually and neutrally in 3-4 sentences. "
    "Do not add or assume information that is not stated in the letter. "
    "Do not say whether the loan should be approved or rejected."
)

def SUMMARY_PROMPT_V2(letter):
    return f"Summarize this loan application:\n\n{letter}"


print("\nV2 - L002:")
answer, usage = ask_llm(
    SUMMARY_PROMPT_V2(LETTERS["L002"]),
    system_prompt=SUMMARY_SYSTEM,
    temperature=0
)
print(answer)

print("\nV2 - L006:")
answer, usage = ask_llm(
    SUMMARY_PROMPT_V2(LETTERS["L006"]),
    system_prompt=SUMMARY_SYSTEM,
    temperature=0
)
print(answer)

L002:
Kwame Boateng, a commercial driver in Kumasi, urgently needs GHS 25,000 to repair his vehicle's engine and pay personal debts. He's struggling due to slow business, but expects it to improve after the festive season. He has no collateral, but promises to repay the loan as soon as possible.

L006:
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no experience or collateral, but claims to be "business-minded" and promises to repay the loan within a year when his businesses are successful, relying on his personal trustworthiness.

V2 - L002:
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but he expects it to improve after the festive season. He does not currently have collateral to offer, but is

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** [
1. What concrete problems did V1’s output have that V2 fixed? Quote examples.
V1 added some assumptions that were not directly stated in the letters. For example, for L002 it said Kwame was “struggling due to slow business,” even though the letter only said business had been slow. For L006, V1 said Kofi “has no experience,” but the letter only said he had not started those businesses yet. V2 was more careful and stayed closer to the information in the letters.

2. Why is “no invented details” an essential instruction in this application? What is this failure mode called in the LLM literature?
It is important because the system is being used to support loan decisions, so invented information could unfairly influence the loan officer. If the model adds facts that are not in the application, it can make the applicant look stronger or weaker than they really are. This type of error is called hallucination.]

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [14]:
import json
import pandas as pd

EXTRACT_SYSTEM = "You extract loan application details and return only valid JSON."

def EXTRACT_PROMPT(letter):
    return f"""
Extract the information from the loan application below.

Return ONLY a JSON object with exactly these keys:
- applicant_name: string
- amount_ghs: number
- purpose: string
- monthly_profit_ghs: number or null
- has_collateral_or_guarantor: boolean
- repayment_months: number or null

If a field is not stated, use null. Do not guess.

Example:
Letter:
My name is Ama Serwaa. I need GHS 6,000 to buy a new oven for my bakery.
I make about GHS 1,200 profit each month. My brother will guarantee the loan.
I plan to repay in 10 months.

JSON:
{{
    "applicant_name": "Ama Serwaa",
    "amount_ghs": 6000,
    "purpose": "buy a new oven for bakery",
    "monthly_profit_ghs": 1200,
    "has_collateral_or_guarantor": true,
    "repayment_months": 10
}}

Loan application:
{letter}
"""


def extract_fields(letter_text):
    raw, usage = ask_llm(
        EXTRACT_PROMPT(letter_text),
        system_prompt=EXTRACT_SYSTEM,
        temperature=0
    )

    cleaned = raw.strip()
    cleaned = cleaned.replace("```json", "")
    cleaned = cleaned.replace("```JSON", "")
    cleaned = cleaned.replace("```", "")
    cleaned = cleaned.strip()

    try:
        return json.loads(cleaned)

    except json.JSONDecodeError:
        print("Could not parse JSON")
        print(raw)
        return None


results = {}

for letter_id, letter_text in LETTERS.items():
    results[letter_id] = extract_fields(letter_text)


df = pd.DataFrame.from_dict(results, orient="index")

df


,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
L002,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
L004,Yaw Owusu,12000,poultry farm at Nsawam for feed and 500 new la...,1500.0,True,18.0
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** [
1.The example should not come from the six letters because the model should be tested on information it has not already seen in the example. Using one of the actual letters could make the results look better because the model may copy the pattern instead of properly extracting the information.

2.The instruction tells the model not to make up information when something is missing from a letter. Without it, the model may try to fill missing fields with values that sound reasonable even though they were never stated. For example, some applicants did not state their monthly profit, so the correct result should be null rather than an estimated amount.

3.Temperature 0 is suitable for extraction because we want the model to be consistent and follow the same format each time. For creative tasks, a higher temperature can be useful because it allows the model to produce more varied ideas.

]

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [16]:
BRIEF_SYSTEM = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Your job is to support the officer, not make the final decision. "
    "Use only facts directly stated in the letter or extracted data. "
    "Do not make assumptions about experience, income, risk, or ability unless stated. "
    "Do not invent any information. "
    "Do not say approve or reject. The final decision must be made by a human."
)


def BRIEF_PROMPT(letter, extracted):
    return f"""
Review the loan application and extracted information below.

Give your response using these four sections:

1. Strengths
2. Risks / red flags
3. Missing information
4. Suggested next step

Only use information stated in the letter or extracted data.
Do not invent or assume information.
Do not approve or reject the loan.

Letter:
{letter}

Extracted information:
{extracted}
"""


briefs = {}

for letter_id, letter_text in LETTERS.items():

    prompt = BRIEF_PROMPT(letter_text, results[letter_id])

    answer, usage = ask_llm(
        prompt,
        system_prompt=BRIEF_SYSTEM,
        temperature=0
    )

    briefs[letter_id] = answer


print("L001:")
print(briefs["L001"])

print("\nL002:")
print(briefs["L002"])

print("\nL006:")
print(briefs["L006"])

L001:
## Step 1: Identify the strengths of the loan application
The applicant, Akosua Mensah, has a long history of selling provisions at Makola Market, indicating stability and experience in her business. She has a steady monthly profit of GHS 900, which suggests a consistent income stream. Additionally, she has saved GHS 2,500 over two years through the susu scheme without missing any contributions, demonstrating her ability to manage savings and potentially repay a loan. She also has a guarantor, her sister, who is a teacher, which could provide an added layer of security for the loan.

## Step 2: Determine the risks or red flags associated with the loan application
One potential risk is that the applicant is looking to expand into frozen foods, which may require additional skills or knowledge and could pose a risk if not managed properly. However, this is not explicitly stated as a risk in the provided information. The repayment plan of GHS 450 over 20 months needs to be evaluated 

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** [
1.Yes, the system was able to separate stronger and weaker applications. L003 has clearer financial information, an existing registered business, collateral, and a repayment plan, while L006 has not started the proposed businesses, has no collateral, and depends on future success to repay the loan. The system generally identified these differences correctly.

2.Practically, the model may misunderstand information or make incorrect assumptions, so a loan officer should review the application before any final decision is made. Ethically, allowing the model to make the final decision could unfairly affect applicants, especially if the model is biased or if important information is missing. The final decision should remain with a human.]

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [e443739e141bcbaf725041541a063fccccd8c73d]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [17]:
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

rows = []

for field in fields:
    correct = 0
    row = {"field": field}

    for letter_id in ["L001", "L003", "L006"]:
        prediction = results[letter_id].get(field)
        actual = GOLD[letter_id][field]

        if field == "applicant_name":
            match = str(prediction).lower() == str(actual).lower()
        else:
            match = prediction == actual

        row[letter_id] = "✓" if match else "✗"

        if match:
            correct += 1

    row["accuracy"] = f"{correct}/3"
    rows.append(row)

accuracy_table = pd.DataFrame(rows)

accuracy_table

,field,L001,L003,L006,accuracy
0,applicant_name,✓,✓,✓,3/3
1,amount_ghs,✓,✓,✓,3/3
2,purpose,✗,✗,✗,0/3
3,monthly_profit_ghs,✓,✓,✓,3/3
4,has_collateral_or_guarantor,✓,✓,✓,3/3
5,repayment_months,✓,✓,✓,3/3


### Part 4.2 — Reliability: is the system consistent?

In [ ]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.